## Conexão dos Notebooks

<a href="https://colab.research.google.com/github/Yuoneeh/Projeto_Redes_EstDados/blob/main/notebooks/isaque/Tarefa5_Arvore_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
from pathlib import Path

# Detecta se está no Google Colab ou em ambiente local
try:
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ_PROJETO = Path("/content/drive/MyDrive/Projeto interdisciplinar")
except (ImportError, ModuleNotFoundError):
    RAIZ_PROJETO = Path(__file__).resolve().parents[2] if "__file__" in locals() else Path.cwd().parents[1]

# Configura o caminho absoluto de PROC
PROC = RAIZ_PROJETO / "data" / "processed"

## Árvore Final

In [ ]:
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score

SEED = 42
CLASSES = ["OK", "RISCO", "FALHA"]

df = pd.read_csv(PROC / "dataset_final_treino.csv")
FEATURES = [
    "z_robusto", "aumento_pct", "jitter_relativo", "perda_pacotes",
    "timeout_atual", "n5_timeout", "n5_aumento80", "n5_risco"
]

X = df[FEATURES].fillna({"z_robusto": 0.0, "aumento_pct": 0.0, "jitter_relativo": 1.0})
y = df["status"]

# 1. Carregar a árvore escolhida na Tarefa 4 e medir o TESTE TEMPORAL UMA ÚNICA VEZ
arvore_final = joblib.load(PROC / "arvore_ajustada_tarefa4.joblib")

teste_mask = df["bloco"] == "teste"
X_teste, y_teste = X[teste_mask], y[teste_mask]
pred_teste = arvore_final.predict(X_teste)

print("=== 2. TESTE TEMPORAL (MEDIDO UMA VEZ) ===")
print(f"F1 Macro no Teste: {f1_score(y_teste, pred_teste, average='macro', labels=CLASSES, zero_division=0):.4f}")
print(f"Recall de FALHA no Teste: {recall_score(y_teste, pred_teste, labels=['FALHA'], average='macro', zero_division=0):.4f}")
print("\nMatriz 3x3 em contagem (Teste):")
print(pd.DataFrame(confusion_matrix(y_teste, pred_teste, labels=CLASSES),
                   index=[f"Real_{c}" for c in CLASSES], columns=[f"Pred_{c}" for c in CLASSES]))
print("\nRelatório completo no Teste:")
print(classification_report(y_teste, pred_teste, labels=CLASSES, zero_division=0))

# ==============================================================================
# 3. FLUXO QUE A ÁRVORE NÃO VIU (PROTOCOLO B - HOLD-OUT DE FLUXO INTEIRO)
# ==============================================================================
# Separamos 2 fluxos inteiros (1 curto e 1 longo) que ficam 100% fora do treino
medianas_fluxo = df.groupby("fluxo_id")["mediana"].first().sort_values()
fluxos_novos = [medianas_fluxo.index[0], medianas_fluxo.index[-1]]

treino_sem_novos = (df["bloco"] == "treino") & (~df["fluxo_id"].isin(fluxos_novos))
avaliacao_novos = df["fluxo_id"].isin(fluxos_novos)

arvore_protocolo_b = DecisionTreeClassifier(**arvore_final.get_params())
arvore_protocolo_b.fit(X[treino_sem_novos], y[treino_sem_novos])

pred_novos = arvore_protocolo_b.predict(X[avaliacao_novos])
print("\n=== 3. FLUXO QUE A ÁRVORE NÃO VIU (BASELINE PRÓPRIO DO PERÍODO A) ===")
print("Fluxos separados fora do treino:", fluxos_novos)
print(classification_report(y[avaliacao_novos], pred_novos, labels=CLASSES, zero_division=0))